### **Creating a ML Model**

Predicting fetus health given various features.

### **Import Libraries**

In [7]:
import pandas as pd                  # Pandas
import numpy as np                   # Numpy
from matplotlib import pyplot as plt # Matplotlib
import seaborn as sns                # Seaborn

# Package to implement Decision Tree Model
import sklearn
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, VotingClassifier


# Package for data partitioning
from sklearn.model_selection import train_test_split

# Package to visualize Decision Tree
from sklearn import tree

# Package for generating confusion matrix
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Package for generating classification report
from sklearn.metrics import classification_report

# Module to save and load Python objects to and from files
import pickle 

%matplotlib inline

# Display inline plots as vector-based (svg)
%config InlineBackend.figure_formats = ['svg']

In [8]:
df = pd.read_csv('Application_Data.csv')
df.head()

,Applicant_ID,Applicant_Gender,Owned_Car,Owned_Realty,Total_Children,Total_Income,Income_Type,Education_Type,Family_Status,Housing_Type,...,Owned_Work_Phone,Owned_Phone,Owned_Email,Job_Title,Total_Family_Members,Applicant_Age,Years_of_Working,Total_Bad_Debt,Total_Good_Debt,Status
0,5008806,M,1,1,0,112500,Working ...,Secondary / secondary special ...,Married ...,House / apartment ...,...,0,0,0,Security staff ...,2,59,4,0,30,1
1,5008808,F,0,1,0,270000,Commercial associate ...,Secondary / secondary special ...,Single / not married ...,House / apartment ...,...,0,1,1,Sales staff ...,1,53,9,0,5,1
2,5008809,F,0,1,0,270000,Commercial associate ...,Secondary / secondary special ...,Single / not married ...,House / apartment ...,...,0,1,1,Sales staff ...,1,53,9,0,5,1
3,5008810,F,0,1,0,270000,Commercial associate ...,Secondary / secondary special ...,Single / not married ...,House / apartment ...,...,0,1,1,Sales staff ...,1,53,9,0,27,1
4,5008811,F,0,1,0,270000,Commercial associate ...,Secondary / secondary special ...,Single / not married ...,House / apartment ...,...,0,1,1,Sales staff ...,1,53,9,0,39,1


In [9]:
# Dropping null values
df.dropna(inplace = True)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25128 entries, 0 to 25127
Data columns (total 21 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   Applicant_ID          25128 non-null  int64 
 1   Applicant_Gender      25128 non-null  object
 2   Owned_Car             25128 non-null  int64 
 3   Owned_Realty          25128 non-null  int64 
 4   Total_Children        25128 non-null  int64 
 5   Total_Income          25128 non-null  int64 
 6   Income_Type           25128 non-null  object
 7   Education_Type        25128 non-null  object
 8   Family_Status         25128 non-null  object
 9   Housing_Type          25128 non-null  object
 10  Owned_Mobile_Phone    25128 non-null  int64 
 11  Owned_Work_Phone      25128 non-null  int64 
 12  Owned_Phone           25128 non-null  int64 
 13  Owned_Email           25128 non-null  int64 
 14  Job_Title             25128 non-null  object
 15  Total_Family_Members  25128 non-null

In [10]:
# Rename columns
df['Status'] = df['Status'].map({
    0: 'Rejected',
    1: 'Approved'
})

# Distribution of Fetal Health Class column
df['Status'].value_counts(normalize = True)

Status
Approved    0.995185
Rejected    0.004815
Name: proportion, dtype: float64

### **Select Input and Output Features**

In [11]:
# Output column for prediction
output = df['Status']

# Drop columns you don’t want as features
features = df.drop(columns=['Applicant_ID', 'Status', 'Owned_Mobile_Phone'])

# Or explicitly select the input features you want
features = df[['Applicant_Gender', 'Owned_Car', 'Owned_Realty',
               'Total_Children', 'Total_Income', 'Income_Type', 'Education_Type',
               'Family_Status', 'Housing_Type',
               'Owned_Work_Phone', 'Owned_Phone', 'Owned_Email', 'Job_Title',
               'Total_Family_Members', 'Applicant_Age', 'Years_of_Working',
               'Total_Bad_Debt', 'Total_Good_Debt']]

# Strip whitespace from all object (string) columns
for col in features.select_dtypes(include=['object']).columns:
    features[col] = features[col].str.strip()


C:\Users\krist\AppData\Local\Temp\ipykernel_35636\3436128255.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  features[col] = features[col].str.strip()


In [12]:
# Now one-hot encode categorical variables
cat_var = ["Applicant_Gender", "Owned_Car", "Owned_Realty", "Income_Type", "Education_Type", "Family_Status", "Housing_Type", "Owned_Work_Phone", "Owned_Phone", "Owned_Email", "Job_Title"]
features_encoded = pd.get_dummies(features, columns=cat_var, drop_first=False)

features_encoded.head()

,Total_Children,Total_Income,Total_Family_Members,Applicant_Age,Years_of_Working,Total_Bad_Debt,Total_Good_Debt,Applicant_Gender_F,Applicant_Gender_M,Owned_Car_0,...,Job_Title_Laborers,Job_Title_Low-skill Laborers,Job_Title_Managers,Job_Title_Medicine staff,Job_Title_Private service staff,Job_Title_Realty agents,Job_Title_Sales staff,Job_Title_Secretaries,Job_Title_Security staff,Job_Title_Waiters/barmen staff
0,0,112500,2,59,4,0,30,False,True,False,...,False,False,False,False,False,False,False,False,True,False
1,0,270000,1,53,9,0,5,True,False,True,...,False,False,False,False,False,False,True,False,False,False
2,0,270000,1,53,9,0,5,True,False,True,...,False,False,False,False,False,False,True,False,False,False
3,0,270000,1,53,9,0,27,True,False,True,...,False,False,False,False,False,False,True,False,False,False
4,0,270000,1,53,9,0,39,True,False,True,...,False,False,False,False,False,False,True,False,False,False


### **Data Partitioning**

In [13]:
train_X, test_X, train_y, test_y = train_test_split(features_encoded, output, test_size = 0.2, random_state = 1) 

## **Prediction Modeling using AdaBoost**

In [14]:
# Defining prediction model
from sklearn.ensemble import AdaBoostClassifier


ab_clf = AdaBoostClassifier(random_state = 0)

# Fitting model on training data
ab_clf.fit(train_X, train_y)

AdaBoostClassifier(random_state=0)

AB Confusion Matrix

In [15]:
# Predictions on training set
y_pred_train_ab = ab_clf.predict(train_X)

# Now generate confusion matrix
cm = confusion_matrix(train_y, y_pred_train_ab, labels = ab_clf.classes_)
disp = ConfusionMatrixDisplay(confusion_matrix = cm, display_labels=ab_clf.classes_)

# Specify figure size
fig, ax = plt.subplots(figsize = (5, 5))
plt.rcParams.update({'font.size': 12})

# Display Confusion Matrix
disp.plot(cmap = 'Reds', ax = ax);

In [16]:
# Predictions on test set
y_pred_ab = ab_clf.predict(test_X)

# Now generate confusion matrix
cm = confusion_matrix(test_y, y_pred_ab, labels = ab_clf.classes_)
disp = ConfusionMatrixDisplay(confusion_matrix = cm, display_labels=ab_clf.classes_)

# Specify figure size
fig, ax = plt.subplots(figsize = (5, 5))
plt.rcParams.update({'font.size': 12})

# Display Confusion Matrix
disp.plot(cmap = 'Reds', ax = ax)

# Save as SVG
plt.savefig("ab_confusion_mat.svg", bbox_inches = 'tight');

AB Classification Report

In [17]:
def make_report(y_true, y_pred):
    report = classification_report(y_true, y_pred, output_dict=True)
    report_df = pd.DataFrame(report).T
    report_df = report_df.round(2)
    return report_df

# Example for Voting model
voting_report_df = make_report(test_y, y_pred_ab)

# Display with gradient
voting_report_df.style.background_gradient(cmap='Reds')
voting_report_df.to_csv("ab_class_report.csv")

AB Feature Importance

In [18]:
# Storing importance values from the trained model
importance = ab_clf.feature_importances_

# Storing feature importance as a dataframe
feature_imp = pd.DataFrame(list(zip(train_X.columns, importance)),
               columns = ['Feature', 'Importance'])

feature_imp = feature_imp.sort_values('Importance', ascending = False).reset_index(drop = True)

# Bar plot
plt.figure(figsize = (10, 5))
plt.barh(feature_imp['Feature'], feature_imp['Importance'], color = ['red', 'bisque'])

plt.xticks(fontsize=5)                      # x-axis tick labels
plt.yticks(fontsize=5) 
plt.xlabel("Importance")
plt.ylabel("Input Feature")
plt.title('Which features are the most important for price range prediction?') 
plt.tight_layout()
plt.savefig("ab_feature_imp.svg");

Save AdaBoost Model

In [19]:
# Pickle file: saving the trained DT model
# Creating the file where we want to write the model
ab_pickle = open('adaboost.pickle', 'wb') 

# Write DT model to the file
pickle.dump(ab_clf, ab_pickle) 

# Close the file
ab_pickle.close() 